In [9]:
# FEATURE ENGINEERING

import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\Supply Chain Shipment Performance Analytics\data\cleaned\supply_chain_master.csv',
                 parse_dates=['order_date','ship_date'])

# ─── FEATURE 1: Schedule Adherence Ratio ────────────────────
# Business: How efficiently did we use the promised window?
# > 1 = ran over; < 1 = delivered faster than promised

df['schedule_adherence_ratio'] = df['days_for_shipping_real'] / df['days_for_shipment_scheduled'].replace(0, np.nan)

# ─── FEATURE 2: Revenue per Unit ────────────────────────────
# Business: Are high-value items more profitable per unit?

df['revenue_per_unit'] = df['sales_per_customer']/ df['order_item_quantity'].replace(0, np.nan)

# ─── FEATURE 3: Is Late (Binary Flag) ───────────────────────

df['is_late'] = (df['delay_days']>0).astype(int)

# ─── FEATURE 4: Order Season ────────────────────────────────
# Business: Do delays cluster in Q4 (holiday season)?

def get_season(month):
    if month in [12,1,2]: return 'Winter'
    elif month in [3,4,5]: return 'Spring'
    elif month in [6,7,8]: return 'Summer'
    else: return 'Fall'

df['order_season'] = df['order_date'].dt.month.apply(get_season)

# ─── FEATURE 5: Ship Mode Encoded ───────────────────────────

ship_mode_map = {
    'Same Day' : 1,
    'First Class' : 2,
    'Second Class' : 3,
    'Standard Class' : 4
}
df['ship_mode_encoded'] = df['shipping_mode'].map(ship_mode_map).fillna(3)

# ─── FEATURE 6: High Value Order ────────────────────────────
# Business: Do premium orders get prioritized in ops?

revenue_75th = df['sales_per_customer'].quantile(0.75)
df['is_high_value'] = (df['sales_per_customer'] > revenue_75th).astype(int)

# ─── FEATURE 7: Market Encoded ──────────────────────────────

market_map = {'Europe' : 1, 'Usca' : 2, 'Latam' : 3, 'Pacific Asia' : 4, 'Africa' : 5}
df['market_encoded'] = df['market'].map(market_map).fillna(3)

# ─── Summary ────────────────────────────────────────────────

new_features = [
    'schedule_adherence_ratio', 'revenue_per_unit', 'is_late',
    'order_season', 'ship_mode_encoded', 'is_high_value', 'market_encoded'
]
print("New features created:")
print(df[new_features].describe())

df.to_csv('../data/cleaned/supply_chain_features.csv', index=False)
print("\n✅ Feature-enriched dataset saved.")

New features created:
       schedule_adherence_ratio  revenue_per_unit        is_late  \
count             170782.000000     180519.000000  180519.000000   
mean                   1.367476        126.908898       0.572793   
std                    0.643969        126.299501       0.494674   
min                    0.500000          7.490000       0.000000   
25%                    0.750000         45.000000       0.000000   
50%                    1.250000         59.990002       1.000000   
75%                    2.000000        179.990005       1.000000   
max                    3.000000       1939.989990       1.000000   

       ship_mode_encoded  is_high_value  market_encoded  
count      180519.000000  180519.000000   180519.000000  
mean            3.334945       0.249719        2.657571  
std             0.924419       0.432851        1.274785  
min             1.000000       0.000000        1.000000  
25%             3.000000       0.000000        1.000000  
50%             4